# 📊 HR-Pulse — EDA Complet 
## Analyse Exploratoire des Données · `jobs_dataset.csv`

---

> **Objectif :** Comprendre les données avant d'entraîner un modèle ML qui prédit le salaire moyen d'un poste Data Science.   
> **Dataset :** 672 offres d'emploi (Data Science / ML) avec salaires, secteurs, tailles d'entreprise…

### 🗺️ Plan du notebook
| Étape | Contenu |
|-------|---------|
| 1 | Installation & imports |
| 2 | Chargement et premier regard |
| 3 | Nettoyage des valeurs aberrantes (`-1`) |
| 4 | Création de la variable cible (Salary) |
| 5 | Analyse des valeurs manquantes |
| 6 | Distribution des salaires |
| 7 | Analyse des variables catégorielles |
| 8 | Analyse des variables numériques & corrélations |
| 9 | Feature Engineering |
| 10 | Préparation finale pour le modèle ML |


---
## ✅ Étape 1 — Installation des bibliothèques & Imports

### Pourquoi ces bibliothèques ?
| Bibliothèque | Rôle |
|---|---|
| `pandas` | Manipuler les tableaux de données (DataFrame) |
| `numpy` | Calculs mathématiques sur des tableaux |
| `matplotlib` | Créer des graphiques de base |
| `seaborn` | Graphiques statistiques plus beaux |
| `scikit-learn` | Outils ML : encodage, split, modèles |


In [29]:
# Télècharger les biblios nécessaires
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

print(" Toutes les bibliothèques sont importées avec succès !")


 Toutes les bibliothèques sont importées avec succès !


---
## ✅ Étape 2 — Chargement des données & Premier regard

### Pourquoi cette étape ?
Avant tout, on veut répondre à ces questions :
- Combien de lignes (offres) et de colonnes (informations) ?
- Quel type de données (texte, nombre, date) ?
- Les premières lignes semblent-elles correctes ?


In [30]:
# ── Charger le CSV ──
df = pd.read_csv('../data/jobs_dataset.csv')

# ── Dimensions du dataset ──
print(f" Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f" Colonnes   : {df.columns.tolist()}")

 Dimensions : 672 lignes × 15 colonnes
 Colonnes   : ['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors']


In [31]:
# ── Aperçu des 5 premières lignes ──
df.head()


,index,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,0,Sr Data Scientist,$137K-$171K (Glassdoor est.),Description\n\nThe Senior Data Scientist is re...,3.1,Healthfirst\n3.1,"New York, NY","New York, NY",1001 to 5000 employees,1993,Nonprofit Organization,Insurance Carriers,Insurance,Unknown / Non-Applicable,"EmblemHealth, UnitedHealth Group, Aetna"
1,1,Data Scientist,$137K-$171K (Glassdoor est.),"Secure our Nation, Ignite your Future\n\nJoin ...",4.2,ManTech\n4.2,"Chantilly, VA","Herndon, VA",5001 to 10000 employees,1968,Company - Public,Research & Development,Business Services,$1 to $2 billion (USD),-1
2,2,Data Scientist,$137K-$171K (Glassdoor est.),Overview\n\n\nAnalysis Group is one of the lar...,3.8,Analysis Group\n3.8,"Boston, MA","Boston, MA",1001 to 5000 employees,1981,Private Practice / Firm,Consulting,Business Services,$100 to $500 million (USD),-1
3,3,Data Scientist,$137K-$171K (Glassdoor est.),JOB DESCRIPTION:\n\nDo you have a passion for ...,3.5,INFICON\n3.5,"Newton, MA","Bad Ragaz, Switzerland",501 to 1000 employees,2000,Company - Public,Electrical & Electronic Manufacturing,Manufacturing,$100 to $500 million (USD),"MKS Instruments, Pfeiffer Vacuum, Agilent Tech..."
4,4,Data Scientist,$137K-$171K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee"


In [37]:
# ── Types de données de chaque colonne ──
print(" Types de données :\n")
print(df.dtypes)

 Types de données :

index                  int64
Job Title                str
Salary Estimate          str
Job Description          str
Rating               float64
Company Name             str
Location                 str
Headquarters             str
Size                     str
Founded                int64
Type of ownership        str
Industry                 str
Sector                   str
Revenue                  str
Competitors              str
dtype: object


In [38]:
# ── Statistiques générales pour les colonnes numériques ──
# count = nb valeurs | mean = moyenne | std = écart-type | min/max = bornes
df.describe()

,index,Rating,Founded
count,672.000000,672.000000,672.000000
mean,335.500000,3.518601,1635.529762
std,194.133974,1.410329,756.746640
min,0.000000,-1.000000,-1.000000
25%,167.750000,3.300000,1917.750000
50%,335.500000,3.800000,1995.000000
75%,503.250000,4.300000,2009.000000
max,671.000000,5.000000,2019.000000


### 🔍 Ce qu'on observe
- **672 offres** d'emploi, **15 colonnes**
- `Salary Estimate` est un **texte** (`"$137K-$171K"`) → il faudra le convertir en nombre
- `Rating` et `Founded` ont un **minimum à -1** → c'est un code pour "donnée manquante" dans ce dataset
- `Job Description` contient de **longs textes** → utile pour le NER, mais pas directement pour le modèle numérique


In [ ]:
# Valeurs manquantes par colonne
print(df.isnull().sum()) 

index                0
Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
dtype: int64


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              672 non-null    int64  
 1   Job Title          672 non-null    str    
 2   Salary Estimate    672 non-null    str    
 3   Job Description    672 non-null    str    
 4   Rating             672 non-null    float64
 5   Company Name       672 non-null    str    
 6   Location           672 non-null    str    
 7   Headquarters       672 non-null    str    
 8   Size               672 non-null    str    
 9   Founded            672 non-null    int64  
 10  Type of ownership  672 non-null    str    
 11  Industry           672 non-null    str    
 12  Sector             672 non-null    str    
 13  Revenue            672 non-null    str    
 14  Competitors        672 non-null    str    
dtypes: float64(1), int64(2), str(12)
memory usage: 78.9 KB


In [14]:
df.describe()

,index,Rating,Founded
count,672.000000,672.000000,672.000000
mean,335.500000,3.518601,1635.529762
std,194.133974,1.410329,756.746640
min,0.000000,-1.000000,-1.000000
25%,167.750000,3.300000,1917.750000
50%,335.500000,3.800000,1995.000000
75%,503.250000,4.300000,2009.000000
max,671.000000,5.000000,2019.000000
